In [158]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

# Lectura Base de datos

## Inspeccionando libro de Excel

In [159]:
xl = pd.ExcelFile('entrada\CLIENTES ANSATTV.xlsx')

In [160]:
xl.sheet_names

['CLIENTES NOROCCIDENTE', 'CLIENTES SAN ANTONIO']

In [161]:
df_preview = pd.read_excel(xl, nrows=1)
df_preview

,CLIENTES NOROCCIDENTE,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,CI,CONTRATO,FECHA,Nombre del Usuario,PLAN,TELEFONO,DIRECCION,IP


In [162]:
df_noroccidente = pd.read_excel(xl, sheet_name='CLIENTES NOROCCIDENTE', header=1)
#df_noroccidente.head(2)

In [163]:
df_noroccidente.shape

(487, 8)

In [164]:
df_noroccidente.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CI                  487 non-null    object
 1   CONTRATO            487 non-null    object
 2   FECHA               487 non-null    object
 3   Nombre del Usuario  487 non-null    object
 4   PLAN                487 non-null    object
 5   TELEFONO            487 non-null    object
 6   DIRECCION           486 non-null    object
 7   IP                  487 non-null    object
dtypes: object(8)
memory usage: 30.6+ KB


In [165]:
df_preview = pd.read_excel(xl, nrows=3, sheet_name='CLIENTES SAN ANTONIO')
#df_preview

In [166]:
df_san_antonio = pd.read_excel(xl, sheet_name='CLIENTES SAN ANTONIO', header=1)
#df_san_antonio.head(2)

In [167]:
df_san_antonio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CI/RUC     448 non-null    int64         
 1   #CONTRATO  448 non-null    int64         
 2   FECHA      448 non-null    datetime64[ns]
 3   NOMBRE     0 non-null      float64       
 4   PLAN       448 non-null    object        
 5   TELEFONO   448 non-null    object        
 6   DIRECCION  448 non-null    object        
 7   IP         448 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 28.1+ KB


## Noroccidente

In [168]:
df_noroccidente.shape

(487, 8)

In [169]:
df_noroccidente.isna().sum()

CI                    0
CONTRATO              0
FECHA                 0
Nombre del Usuario    0
PLAN                  0
TELEFONO              0
DIRECCION             1
IP                    0
dtype: int64

In [170]:
np.where(df_noroccidente.isna())

(array([216]), array([6]))

In [171]:
#df_noroccidente.iloc[216]

In [172]:
conteo_contratos = df_noroccidente['CONTRATO'].value_counts()

In [173]:
contratos_repetidos = conteo_contratos[conteo_contratos > 1]

In [174]:
contratos_repetidos

CONTRATO
CON 0000919    2
Name: count, dtype: int64

In [175]:
#df_noroccidente[ df_noroccidente['CONTRATO'].str.contains('CON 0000919') ]

Mismo contrato y cédula con direcciones IP **distintas**

In [176]:
contratos_duplicados = df_noroccidente[ df_noroccidente['CONTRATO'].str.contains('CON 0000919') ]

In [177]:
with pd.ExcelWriter('Proc_noroccidente.xlsx', engine='openpyxl') as writer:
    contratos_duplicados.to_excel(writer, sheet_name='Duplicados_Contratos', index=False)

In [178]:
conteo_ci = df_noroccidente['CI'].value_counts()

In [179]:
ci_repeditos = conteo_ci[conteo_ci > 1]

In [180]:
#ci_repeditos

In [181]:
len(ci_repeditos)

21

In [182]:
ci_repeditos.sum()

np.int64(49)

In [183]:
#df_noroccidente[ df_noroccidente['CI'] == '']

In [184]:
#for ci, cantidad in ci_repeditos.items():
#    print (ci)

números de cédulas repetidas

In [185]:
lista_registros_ci = []
for ci, cantidad in ci_repeditos.items():
    df_ci = df_noroccidente[df_noroccidente['CI'] == ci]
    lista_registros_ci.append(df_ci)


In [186]:
duplicados_ci = pd.concat(lista_registros_ci, ignore_index=True)

In [187]:
duplicados_ci.shape

(49, 8)

In [188]:
with pd.ExcelWriter('Proc_noroccidente.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    duplicados_ci.to_excel(writer, sheet_name='Duplicados_CI', index=False)

In [189]:
pd.pivot_table(
    df_noroccidente,
    values='CONTRATO',
    index='PLAN',
    aggfunc='count'
)

,CONTRATO
PLAN,
250 MB,440
300 MB,34
350 MB,10
400 MB,1
450 MB,2


In [190]:
#duplicados_ci.head(3)

In [191]:
pivot_completo = (duplicados_ci
    .groupby('CI')
    .agg(
        Contratos=('CONTRATO', 'count'),
        IPs=('IP', 'nunique'),
        Telefonos=('TELEFONO', 'nunique'),
        Nombres=('Nombre del Usuario', 'nunique')
    )
    .assign(
        Tipo_Duplicidad=lambda x: 
            np.where(x['Nombres'] > 1, 'Diferentes_nombres',
            np.where((x['IPs'] > 1) & (x['Telefonos'] == 1), 'Mismo_nombre_diferentes_IP',
            np.where((x['IPs'] == 1) & (x['Telefonos'] == 1), 'Registro_duplicado_identico',
            np.where((x['IPs'] > 1) & (x['Telefonos'] > 1), 'Multiples_IP_y_telefonos', 'Patron_mixto'))))
    )
    .sort_values('Contratos', ascending=False)
)

#print(pivot_completo)

In [192]:
with pd.ExcelWriter('Proc_noroccidente.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    pivot_completo.to_excel(writer, sheet_name='Duplicidad_CI', index=True)

In [193]:
#df_noroccidente[ df_noroccidente['CI'] == '' ]

In [194]:
listado_ip = df_noroccidente['IP'].value_counts()

In [195]:
duplicados_ip = listado_ip[ listado_ip > 1 ]

In [196]:
duplicados_ip

Series([], Name: count, dtype: int64)

## San Antonio

In [197]:
df_san_antonio.shape

(448, 8)

In [198]:
df_san_antonio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CI/RUC     448 non-null    int64         
 1   #CONTRATO  448 non-null    int64         
 2   FECHA      448 non-null    datetime64[ns]
 3   NOMBRE     0 non-null      float64       
 4   PLAN       448 non-null    object        
 5   TELEFONO   448 non-null    object        
 6   DIRECCION  448 non-null    object        
 7   IP         448 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 28.1+ KB


In [199]:
df_san_antonio.isna().sum()

CI/RUC         0
#CONTRATO      0
FECHA          0
NOMBRE       448
PLAN           0
TELEFONO       0
DIRECCION      0
IP             0
dtype: int64

In [200]:
conteo_contratos_sa = df_san_antonio['#CONTRATO'].value_counts()

In [201]:
contratos_repetidos_sa = conteo_contratos_sa[conteo_contratos_sa > 1]

In [202]:
contratos_repetidos_sa

#CONTRATO
10397    2
1359     2
1460     2
Name: count, dtype: int64

In [203]:
#df_san_antonio[ df_san_antonio['#CONTRATO'] == 10397 ]

Mismo contrato y cédula con direcciones IP **distintas**

In [204]:
lista_contratos = []
for c, cantidad in contratos_repetidos_sa.items():
    df_c = df_san_antonio[df_san_antonio['#CONTRATO'] == c]
    lista_contratos.append(df_c)

In [205]:
duplicados_c = pd.concat(lista_contratos, ignore_index=True)

In [206]:
#duplicados_c

In [207]:
with pd.ExcelWriter('Proc_san_antonio.xlsx', engine='openpyxl') as writer:
    duplicados_c.to_excel(writer, sheet_name='Duplicados_Contratos', index=False)

In [208]:
df_san_antonio.columns.values

array(['CI/RUC', '#CONTRATO', 'FECHA', 'NOMBRE', 'PLAN', 'TELEFONO',
       'DIRECCION', 'IP'], dtype=object)

In [209]:
conteo_ci_sa = df_san_antonio['CI/RUC'].value_counts()

In [210]:
ci_repeditos_sa = conteo_ci_sa[conteo_ci_sa > 1]

In [211]:
#ci_repeditos_sa

In [212]:
len(ci_repeditos_sa)

4

In [213]:
ci_repeditos_sa.sum()

np.int64(9)

números de cédulas repetidas

In [214]:
lista_registros_ci_sa = []
for ci, cantidad in ci_repeditos_sa.items():
    df_ci = df_san_antonio[df_san_antonio['CI/RUC'] == ci]
    lista_registros_ci_sa.append(df_ci)


In [215]:
duplicados_ci_sa = pd.concat(lista_registros_ci_sa, ignore_index=True)

In [216]:
duplicados_ci_sa.shape

(9, 8)

In [217]:
#duplicados_ci_sa

In [218]:
with pd.ExcelWriter('Proc_san_antonio.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    duplicados_ci_sa.to_excel(writer, sheet_name='Duplicados_CI', index=False)

In [219]:
pd.pivot_table(
    df_san_antonio,
    values='#CONTRATO',
    index='PLAN',
    aggfunc='count'
)

,#CONTRATO
PLAN,
250Mbps,330
300Mbps,72
350Mbps,29
400Mbps,5
450Mbps,12


In [220]:
pivot_completo_sa = (duplicados_ci_sa
    .groupby('CI/RUC')
    .agg(
        Contratos=('#CONTRATO', 'count'),
        IPs=('IP', 'nunique'),
        Telefonos=('TELEFONO', 'nunique'),
        Nombres=('NOMBRE', 'nunique')
    )
    .assign(
        Tipo_Duplicidad=lambda x: 
            np.where(x['Nombres'] > 1, 'Diferentes_nombres',
            np.where((x['IPs'] > 1) & (x['Telefonos'] == 1), 'Mismo_nombre_diferentes_IP',
            np.where((x['IPs'] == 1) & (x['Telefonos'] == 1), 'Registro_duplicado_identico',
            np.where((x['IPs'] > 1) & (x['Telefonos'] > 1), 'Multiples_IP_y_telefonos', 'Patron_mixto'))))
    )
    .sort_values('Contratos', ascending=False)
)

#print(pivot_completo_sa)

In [221]:
with pd.ExcelWriter('Proc_san_antonio.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    pivot_completo_sa.to_excel(writer, sheet_name='Duplicidad_CI', index=True)

In [222]:
listado_ip_sa = df_san_antonio['IP'].value_counts()

In [223]:
duplicados_ip_sa = listado_ip_sa[ listado_ip_sa > 1 ]

In [224]:
duplicados_ip_sa

Series([], Name: count, dtype: int64)

No se visualizan IPs duplicadas

## JOIN noroccidental y san_antonio